In [1]:
import pandas as pd
import numpy as np
import os
import sqlite3

## Audit the raw data

In [2]:
# 1. Load the three raw tables
# Note: keep_default_na=False ensures empty ticket texts stay empty strings rather than NaN
customers_df = pd.read_csv("data/raw/customers.csv")
orders_df = pd.read_csv("data/raw/orders.csv")
products_df = pd.read_csv("data/raw/products.csv")
tickets_df = pd.read_csv(
    "data/raw/tickets.csv", keep_default_na=False
)

tables_1= {
    "customers.csv": customers_df,
    "orders.csv": orders_df,
}

# 2. General Audit: Shape, Data Types, Missing Counts & Percentages
for name, df in tables_1.items() :
    print(f"\n{'-'*40}\nAUDIT REPORT: {name}\n{'-'*40}")
    print(f"Shape (Rows, Columns): {df.shape}\n")
    print("Data Types & Missing Values:")

    audit_summary = pd.DataFrame(
        {
            "Data Type": df.dtypes,
            "Missing Count": df.isnull().sum(),
            "Missing Percentage (%)": (df.isnull().sum() / len(df)) * 100,
        }
    )
    print(audit_summary.to_string())


----------------------------------------
AUDIT REPORT: customers.csv
----------------------------------------
Shape (Rows, Columns): (1200, 5)

Data Types & Missing Values:
            Data Type  Missing Count  Missing Percentage (%)
customer_id       str              0                0.000000
city              str              0                0.000000
signup_date       str              0                0.000000
age           float64             61                5.083333
gender            str             30                2.500000

----------------------------------------
AUDIT REPORT: orders.csv
----------------------------------------
Shape (Rows, Columns): (13084, 9)

Data Types & Missing Values:
                 Data Type  Missing Count  Missing Percentage (%)
order_id               str              0                0.000000
customer_id            str              0                0.000000
product_id             str              0                0.000000
quantity             int

### Mistakes found in Customer dataset
signup_date column data type = str
age column missing count = 61 Missing Percentage (%) = 5.083333
gender column missing count = 30 Missing Percentage (%) = 2.500000

### Mistakes found in Order dataset
order_ts(timestamp) column data type = str
order_amount column missing count = 53 Missing Percentage (%) = 0.405075
delivery_minutes column missing count = 2541 Missing Percentage (%) = 9.420666 and data type = float


In [4]:
tables_2= {
    "products.csv" : products_df,
    "tickets.csv" : tickets_df,

}
for name, df in tables_2.items() :
    print(f"\n{'-'*40}\nAUDIT REPORT: {name}\n{'-'*40}")
    print(f"Shape (Rows, Columns): {df.shape}\n")
    print("Data Types & Missing Values:")

    audit_summary = pd.DataFrame(
        {
            "Data Type": df.dtypes,
            "Missing Count": df.isnull().sum(),
            "Missing Percentage (%)": (df.isnull().sum() / len(df)) * 100,
        }
    )
    print(audit_summary.to_string())


----------------------------------------
AUDIT REPORT: products.csv
----------------------------------------
Shape (Rows, Columns): (40, 4)

Data Types & Missing Values:
             Data Type  Missing Count  Missing Percentage (%)
product_id         str              0                     0.0
product_name       str              0                     0.0
category           str              0                     0.0
price_inr        int64              0                     0.0

----------------------------------------
AUDIT REPORT: tickets.csv
----------------------------------------
Shape (Rows, Columns): (164, 5)

Data Types & Missing Values:
              Data Type  Missing Count  Missing Percentage (%)
ticket_id           str              0                     0.0
customer_id         str              0                     0.0
order_id            str              0                     0.0
text                str              0                     0.0
true_category       str          

## Order dataset Audit:

In [12]:
print(f"\n{'-'*25}\n--- ORDERS AUDIT ---\n{'-'*25}")
print("Number of rows in ORDERS dataset", orders_df.shape)
total_orders = len(orders_df)
exact_dupes_orders = orders_df.duplicated().sum()
pending_rows = (orders_df["status"].str.lower() == "pending").sum() 
print(f"Total number of pending rows in STATUS columns: {pending_rows}")

distinct_order_ids = (orders_df["order_id"].nunique())
print(f"Number of exact duplicate rows in ORDERS dataset: {exact_dupes_orders}")
print(f"Number of distinct order IDs in ORDERS dataset: {distinct_order_ids}")

non_numeric_amounts = pd.to_numeric(orders_df["order_amount"], errors="coerce").isna().sum()
print(f"Number of non-numeric ORDER AMOUNT columns: {non_numeric_amounts}")

# Delivery minutes checks
# Convert to numeric to safely check negatives and blanks
deliv_numeric = pd.to_numeric(orders_df["delivery_minutes"], errors="coerce")
neg_delivery = (deliv_numeric < 0).sum()
blank_delivery = (
        orders_df["delivery_minutes"].isna()
        | (orders_df["delivery_minutes"] == "")
        | (orders_df["delivery_minutes"].astype(str).str.strip() == "")
    ).sum()

print(f"Number of negative values in DELIVERY MINUTES column: {neg_delivery}")
print(f"Number of blank values in DELIVERY MINUTES column: {blank_delivery}")


-------------------------
--- ORDERS AUDIT ---
-------------------------
Number of rows in ORDERS dataset (13084, 9)
Total number of pending rows in STATUS columns: 656
Number of exact duplicate rows in ORDERS dataset: 119
Number of distinct order IDs in ORDERS dataset: 12309
Number of non-numeric ORDER AMOUNT columns: 120
Number of negative values in DELIVERY MINUTES column: 106
Number of blank values in DELIVERY MINUTES column: 2541


## Orders dataset Clean:

In [6]:
print(f"\n{'-'*30}\n--- ORDERS DATASET CLEAN ---\n{'-'*30}")
# 1. Convert orders_ts to datetime and drop exact duplicates

orders_df["order_ts"] = pd.to_datetime(orders_df["order_ts"], errors = "coerce")
orders_df = orders_df.drop_duplicates()

# 2. Sort the dataframe by order_ts in ascending order, 
# then drop duplicates keeping the *last* occurrence (which is the latest timestamp)
orders_df = (
    orders_df.sort_values(by="order_ts", ascending=True)
    .drop_duplicates(subset=["order_id"], keep="last")
    .reset_index(drop=True)
)

# 3. Convert order_amount to numeric (turn unparseable/invalid entries into NaN)
orders_df["order_amount"] = pd.to_numeric(orders_df["order_amount"], errors="coerce")
print(f"Cleaned rows: {len(orders_df)}")

print(f"\n{'-'*40}\n--- ORDER_AMOUNT COLUMN IN ORDERS ---\n{'-'*40}")

# 1. Merge orders_df with products_df on 'product_id' so every order has its corresponding price_inr
# (Using a left merge ensures you don't lose any order rows)
merged_orders = orders_df.merge(
    products_df[["product_id", "price_inr"]], 
    on="product_id", 
    how="left", 
    suffixes=("", "_prod")
)

# 2. Calculate the order amount using the merged price
calculated_amount = round(
    merged_orders["price_inr"]
    * merged_orders["quantity"]
    * (1 - merged_orders["discount_pct"] / 100),
    2,
)

# 3. Fill the missing order_amounts in the main orders_df
orders_df["order_amount"] = orders_df["order_amount"].fillna(calculated_amount)
print(
    "Remaining missing order amounts after impute:",
    orders_df["order_amount"].isna().sum(),
)

print(f"\n{'-'*40}\n--- DELIVERY_MINUTES COLUMN IN ORDERS---\n{'-'*40}")

# 1. Convert delivery_minutes to numeric, turning unparseable text into NaN
orders_df["delivery_minutes"] = pd.to_numeric(
    orders_df["delivery_minutes"], errors="coerce"
)
# 2. Set negative delivery_minutes to missing (NaN)
orders_df.loc[orders_df["delivery_minutes"] < 0, "delivery_minutes"] = np.nan

# 3. Create a boolean mask for completed orders
completed_mask = (orders_df["status"].astype(str).str.strip().str.lower() == "completed")
cancelled_mask = (orders_df["status"].astype(str).str.strip().str.lower() == "cancelled")

# 4. Calculate median delivery time by product category for completed orders only, and fill missing values
category_medians = (
    orders_df[completed_mask]
    .groupby("product_id")["delivery_minutes"]
    .transform("median")
)
# Apply fillna strictly on completed orders (cancelled orders naturally stay NaN)
orders_df.loc[completed_mask, "delivery_minutes"] = orders_df.loc[
    completed_mask, "delivery_minutes"
].fillna(category_medians)

print(
    "Remaining missing delivery_minutes for completed orders:",
    orders_df[completed_mask]["delivery_minutes"].isna().sum(),
)

orders_df["order_amount"] = pd.to_numeric(
    orders_df["order_amount"], errors="coerce"
)
sum_of_completed_orders = orders_df.loc[completed_mask, "order_amount"].sum()
print(f"Sum of order amount for completed orders: {sum_of_completed_orders}")



------------------------------
--- ORDERS DATASET CLEAN ---
------------------------------
Cleaned rows: 12309

----------------------------------------
--- ORDER_AMOUNT COLUMN IN ORDERS ---
----------------------------------------
Remaining missing order amounts after impute: 0

----------------------------------------
--- DELIVERY_MINUTES COLUMN IN ORDERS---
----------------------------------------
Remaining missing delivery_minutes for completed orders: 0
Sum of order amount for completed orders: 2998668.8000000003


## Customers dataset Audit

In [ ]:
print(f"\n{'-'*25}\n--- CUSTOMER AUDIT---\n{'-'*25}")
# --- Customers Audit ---
age_numeric = pd.to_numeric(customers_df["age"], errors="coerce")
blank_age = (
        customers_df["age"].isna()
        | (customers_df["age"] == "")
        | (customers_df["age"].astype(str).str.strip() == "")
    ).sum()
age_outside = ((age_numeric < 10) | (age_numeric > 100)).sum()

print(f"Number of blank values in AGE column: {blank_age}")
print(f"Number of values outside the 10-100 range in AGE column: {age_outside}")

distinct_cities = customers_df["city"].nunique()
print(f"Number of unique cities in CITY column: {distinct_cities}")

blank_gender = (
        customers_df["gender"].isna()
        | (customers_df["gender"] == "")
        | (customers_df["gender"].astype(str).str.strip().str.lower() == "")
    ).sum()
distinct_genders = (customers_df["gender"].nunique())

print(f"Number of unique raw genders in GENDER column: {distinct_genders}")
print(f"Number of blank values in GENDER column: {blank_gender}")


-------------------------
--- CUSTOMER AUDIT---
-------------------------
Number of values outside the 10-100 range in AGE column: 6
Number of blank values in AGE column: 61
Number of unique cities in CITY column: 24
Number of blank values in GENDER column: 30
Number of unique raw genders in GENDER column: 10


## Customer dataset Clean

In [20]:
print(f"\n{'-'*40}\n--- AGE ON CUSTOMERS TABLE ---\n{'-'*40}")
# 1. Convert age to numeric, turning any invalid types into NaN
customers_df["age"] = pd.to_numeric(customers_df["age"], errors="coerce")

customers_df.loc[(customers_df["age"] < 10) | (customers_df["age"] > 100), "age"] = (
    np.nan
)

# 2. Fill missing ages with the median age of the customer's city using groupby and transform
customers_df["age"] = customers_df["age"].fillna(
    customers_df.groupby("city")["age"].transform("median")
)

print("Total count after fillna", customers_df["age"].count())

missing = customers_df['age'].isna().sum()
out_of_range = (~customers_df['age'].between(18, 60) & customers_df['age'].notna()).sum()

print("Number of Missing ages:", missing)
print("Number of Out of range ages between 18 and 60:", out_of_range)

print(f"\n{'-'*40}\n--- GENDER ON CUSTOMERS TABLE ---\n{'-'*40}")
# 1. First, treat blank spaces, empty strings, and standard NaNs as missing values
customers_df["gender"] = customers_df["gender"].replace(r"^\s*$", np.nan, regex=True)

# 2. Standardize text by filling missing/NaN as 'unknown', converting to string, stripping spaces, and lowercasing
customers_df["gender"] = (
    customers_df["gender"]
    .fillna("unknown")
    .astype(str)
    .str.strip()
    .str.lower()
)

# 3. Create a clean mapping dictionary for all variants
gender_mapping = {
    "male": "male",
    "m": "male",
    "female": "female",
    "f": "female",
    "unknown": "unknown",
}

# 4. Map the variations, catching any unmapped or leftover values into 'unknown'
customers_df["gender"] = customers_df["gender"].map(gender_mapping).fillna("unknown")

# Verify the final counts
print(f"Total count of each gender category after cleaning:")
print(customers_df["gender"].value_counts())

print(f"\n{'-'*40}\n--- CITY ON CUSTOMERS TABLE ---\n{'-'*40}")
customers_df["city"] = customers_df["city"].str.lower().str.strip()

city_mapping = {
    'pune': 'Pune',
    'delhi': 'Delhi',
    'bengaluru': 'Bengaluru',
    'chennai': 'Chennai',
    'hyderabad' : 'Hyderabad',
    'mumbai' : 'Mumbai',
}
customers_df['city'] = customers_df['city'].map(city_mapping)
customers_df["signup_date"] = pd.to_datetime(customers_df["signup_date"], errors='coerce')

print(f"Total count of each city category after cleaning:")
print(customers_df["city"].value_counts())


----------------------------------------
--- AGE ON CUSTOMERS TABLE ---
----------------------------------------
Total count after fillna 1200
Number of Missing ages: 0
Number of Out of range ages between 18 and 60: 0

----------------------------------------
--- GENDER ON CUSTOMERS TABLE ---
----------------------------------------
Total count of each gender category after cleaning:
gender
female     610
male       560
unknown     30
Name: count, dtype: int64

----------------------------------------
--- CITY ON CUSTOMERS TABLE ---
----------------------------------------
Total count of each city category after cleaning:
city
Pune         227
Delhi        210
Bengaluru    197
Chennai      191
Hyderabad    191
Mumbai       184
Name: count, dtype: int64


## 7 SQL Quries:

In [28]:
# 1. Create the 'sql' directory if it doesn't exist
os.makedirs("sql", exist_ok=True)

# 2. Define all 7 SQL queries according to the exact requirements and checkpoints
queries_content = """
-- Q1: WHERE then HAVING (Completed orders > 500, grouped by customer, total > 8000)
SELECT customer_id, SUM(order_amount) AS total_spent
FROM orders
WHERE LOWER(status) = 'completed' AND order_amount > 500
GROUP BY customer_id
HAVING SUM(order_amount) > 8000;

-- Q2: Three-table join (Revenue by city and product category)
SELECT 
    c.city, 
    p.category AS product_category, 
    SUM(o.order_amount) AS total_revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
WHERE LOWER(o.status) = 'completed'
GROUP BY c.city, p.category
ORDER BY total_revenue DESC;

-- Q3: Scalar subquery (Products priced above the average catalogue price)
SELECT product_id, product_name, price_inr
FROM products
WHERE price_inr > (SELECT AVG(price_inr) FROM products);

-- Q4: LEFT JOIN (Customers who have no completed order)
SELECT c.customer_id
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id AND LOWER(o.status) = 'completed'
WHERE o.order_id IS NULL;

-- Q5: Common Table Expressions (Customers whose total completed spend is above the overall average)
WITH customer_spend AS (
    SELECT customer_id, SUM(order_amount) AS total_spend
    FROM orders
    WHERE LOWER(status) = 'completed'
    GROUP BY customer_id
),
avg_spend AS (
    SELECT AVG(total_spend) AS mean_spend FROM customer_spend
)
SELECT cs.customer_id, cs.total_spend
FROM customer_spend cs, avg_spend av
WHERE cs.total_spend > av.mean_spend;

-- Q6: Ranking Window Functions (Rank, Dense Rank, Row Number partitioned by city)
WITH customer_city_spend AS (
    SELECT 
        c.customer_id, 
        c.city, 
        SUM(o.order_amount) AS total_spend
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE LOWER(o.status) = 'completed'
    GROUP BY c.customer_id, c.city
)
SELECT 
    city,
    customer_id,
    total_spend,
    RANK() OVER (PARTITION BY city ORDER BY total_spend DESC) AS rnk,
    DENSE_RANK() OVER (PARTITION BY city ORDER BY total_spend DESC) AS dense_rnk,
    ROW_NUMBER() OVER (PARTITION BY city ORDER BY total_spend DESC) AS row_num
FROM customer_city_spend;

-- Q7: Running Total (Daily completed revenue and cumulative sum)
WITH daily_revenue AS (
    SELECT 
        SUBSTR(order_ts, 1, 10) AS day,
        SUM(order_amount) AS daily_rev
    FROM orders
    WHERE LOWER(status) = 'completed'
    GROUP BY SUBSTR(order_ts, 1, 10)
)
SELECT 
    day,
    daily_rev,
    SUM(daily_rev) OVER (ORDER BY day) AS cumulative_revenue
FROM daily_revenue
ORDER BY day;
"""

# 3. Write the queries file
with open("sql/queries.sql", "w") as f:
    f.write(queries_content.strip())

print("Successfully created sql/queries.sql!\n")

# 4. Connect to database, run each query, and print results + checks
conn = sqlite3.connect("data/instamart.db")

with open("sql/queries.sql", "r") as f:
    sql_script = f.read()

queries = [q.strip() for q in sql_script.split(";") if q.strip()]

for i, query in enumerate(queries, 1):
    print("=" * 60)
    print(f"RUNNING QUERY Q{i}")
    print("=" * 60)

    result_df = pd.read_sql_query(query, conn)

    if i == 1:
        print(f"Result Checkpoint -> Total Customers: {len(result_df)}")

    elif i == 2:
        print(f"Result Checkpoint -> Total Rows: {len(result_df)}")
        print(f"Total Revenue Sum: INR {result_df['total_revenue'].sum():,.2f}")

    elif i == 3:
        print(f"Result Checkpoint -> Total Products: {len(result_df)}")

    elif i == 4:
        print(f"Result Checkpoint -> Total Customers: {len(result_df)}")

    elif i == 5:
        print(f"Result Checkpoint -> Total Customers: {len(result_df)}")
        # Average of ALL customers' spend (not just those above average).
        avg_df = pd.read_sql_query("""
            SELECT AVG(total_spend) AS mean_spend FROM (
                SELECT customer_id, SUM(order_amount) AS total_spend
                FROM orders
                WHERE LOWER(status) = 'completed'
                GROUP BY customer_id
            )
        """, conn)
        print(f"Average Spend: INR {avg_df['mean_spend'].iloc[0]:,.2f}")

    elif i == 6:
        print(f"Result Checkpoint -> Total Rows: {len(result_df)}")

        print("\nTop customer per city (row_num = 1):")
        top = result_df[result_df["row_num"] == 1][["city", "customer_id", "total_spend"]]
        print(top.sort_values("city").to_string(index=False))

        diff = (result_df["rnk"] != result_df["row_num"]).sum()
        print(f"\nRows where RANK() != ROW_NUMBER(): {diff}")
        print("Matches expected 8?", diff == 8)

        print("\nRows affected by ties:")
        print(result_df[result_df["rnk"] != result_df["row_num"]].to_string(index=False))

    elif i == 7:
        print(f"Result Checkpoint -> Total Daily Rows: {len(result_df)}")
        last_cum_val = result_df["cumulative_revenue"].iloc[-1]
        print(f"Last Cumulative Revenue: INR {last_cum_val:,.2f}")

    display(result_df.head())
    print("\n")


Successfully created sql/queries.sql!

RUNNING QUERY Q1
Result Checkpoint -> Total Customers: 30


,customer_id,total_spent
0,C0082,12317.70
1,C0130,9170.40
2,C0147,9141.50
3,C0167,8718.55
4,C0267,12150.25




RUNNING QUERY Q2
Result Checkpoint -> Total Rows: 30
Total Revenue Sum: INR 2,998,668.80


,city,product_category,total_revenue
0,Bengaluru,Electronics,233909.45
1,Hyderabad,Electronics,228913.10
2,Mumbai,Electronics,214738.40
3,Chennai,Electronics,195508.80
4,Pune,Electronics,187629.25




RUNNING QUERY Q3
Result Checkpoint -> Total Products: 11


,product_id,product_name,price_inr
0,P014,Ghee 500ml,340
1,P030,Body Lotion 400ml,350
2,P032,Sunscreen 50g,399
3,P033,Wireless Earbuds,1499
4,P034,Power Bank 10000mAh,1299


... (4 outputs left out)



RUNNING QUERY Q6
Result Checkpoint -> Total Rows: 1175

Top customer per city (row_num = 1):
     city customer_id  total_spend
Bengaluru       C0082     12856.20
  Chennai       C0930     10151.40
    Delhi       C0291     13742.60
Hyderabad       C0743     12730.05
   Mumbai       C0693     14766.55
     Pune       C0426     10377.10

Rows where RANK() != ROW_NUMBER(): 8
Matches expected 8? True

Rows affected by ties:
     city customer_id  total_spend  rnk  dense_rnk  row_num
Bengaluru       C0734      2598.00   99         99      100
Bengaluru       C0723       135.00  179        178      180
   Mumbai       C1053       396.00  149        149      150
   Mumbai       C0608       280.00  160        159      161
     Pune       C1064      3236.95   74         74       75
     Pune       C0057       285.00  189        188      190
     Pune       C0936       285.00  189        188      191
     Pune       C0276       114.00  209        206      210


,city,customer_id,total_spend,rnk,dense_rnk,row_num
0,Bengaluru,C0082,12856.2,1,1,1
1,Bengaluru,C0525,12719.7,2,2,2
2,Bengaluru,C0699,12471.8,3,3,3
3,Bengaluru,C0543,11195.1,4,4,4
4,Bengaluru,C1147,10156.9,5,5,5




RUNNING QUERY Q7
Result Checkpoint -> Total Daily Rows: 183
Last Cumulative Revenue: INR 2,998,668.80


,day,daily_rev,cumulative_revenue
0,2025-04-01,16876.50,16876.50
1,2025-04-02,19854.55,36731.05
2,2025-04-03,28048.35,64779.40
3,2025-04-04,14249.60,79029.00
4,2025-04-05,16031.50,95060.50


## Observation:
Why ties cause RANK() ≠ ROW_NUMBER(): tied customers share one RANK() value while ROW_NUMBER() keeps counting up uniquely, so the two diverge on every tied row after the first.

## Pivot Table

In [40]:
orders_df = pd.read_csv("data/clean/orders.csv")
customers_df = pd.read_csv("data/clean/customers.csv")
products_df = pd.read_csv("data/clean/products.csv")

orders_df["order_amount"] = pd.to_numeric(orders_df["order_amount"], errors="coerce")
cat_col = "category"

# 1. Join the three tables
full = (orders_df
            .merge(customers_df[["customer_id", "city"]], on="customer_id")
            .merge(products_df[["product_id", cat_col]], on="product_id"))

# 2. Keep completed orders only
df = full[full["status"].str.strip().str.lower() == "completed"]

# 3. Pivot table with grand total (city x category)
pt = pd.pivot_table(
        df,
        index="city",
        columns=cat_col,
        values="order_amount",
        aggfunc="sum",
        margins=True,
        margins_name="Total",
        fill_value=0
    )
print(pt.round(2))
pandas_total = pt.loc["Total", "Total"]

# 4. Per-city sum, mean, count (named aggregation)
city_stats = (df.groupby("city")
                    .agg(total_amount=("order_amount", "sum"),
                         mean_amount=("order_amount", "mean"),
                         order_count=("order_amount", "count"))
                    .round(2))
print(city_stats)
print("Pandas grand total: INR", round(pandas_total, 2))

# 5. Compare with the SQL total
con = sqlite3.connect("data/instamart.db")
sql_total = pd.read_sql_query(
        "SELECT SUM(order_amount) AS t FROM orders WHERE status = 'completed'", con
    )["t"].iloc[0]
con.close()

print("SQL total: INR", round(sql_total, 2))
print("Match:", np.isclose(pandas_total, sql_total, atol=0.01))

category       Dairy  Electronics    Grocery  Personal Care     Snacks  \
city                                                                     
Bengaluru   87075.05    233909.45  110915.65       20193.85   71549.40   
Chennai     78301.80    195508.80   93817.45       27675.40   68325.70   
Delhi      117271.70    158549.10  112658.05       23471.50   90879.45   
Hyderabad   83731.00    228913.10  109475.40       24800.05   81670.35   
Mumbai      75209.85    214738.40   78970.15       38018.00   72122.10   
Pune        90349.55    187629.25  108456.60       34275.15   80207.50   
Total      531938.95   1219248.10  614293.30      168433.95  464754.50   

category        Total  
city                   
Bengaluru   523643.40  
Chennai     463629.15  
Delhi       502829.80  
Hyderabad   528589.90  
Mumbai      479058.50  
Pune        500918.05  
Total      2998668.80  
           total_amount  mean_amount  order_count
city                                             
Bengaluru     523

## Pandas cross - check
Pandas Grnad total is : 2998668.8, SQL Query total is : 2998668.8.

Pandas cross-check is succussfull!


## Exploratory Data Analysis (EDA):

In [17]:
# 1. Reload and rebuild the Part A table (same join as Q2)
df_orders    = pd.read_csv("data/clean/orders.csv")
df_customers = pd.read_csv("data/clean/customers.csv")
df_products  = pd.read_csv("data/clean/products.csv")

cat_col = "category"

full = (df_orders
        .merge(df_customers[["customer_id", "city"]], on="customer_id")
        .merge(df_products[["product_id", cat_col]].drop_duplicates("product_id"),
               on="product_id"))

done_b = full[full["status"].str.strip().str.lower() == "completed"].copy()

# 2. Sanity check: must match Q2
print("Rows:", len(done_b), "| Total:", round(done_b["order_amount"].sum(), 2))   # total 2,998,668.80

# 3. Summary statistics of completed order_amount
x = done_b["order_amount"]

q1, q3 = x.quantile(0.25), x.quantile(0.75)
iqr    = q3 - q1
lower  = q1 - 1.5 * iqr
upper  = q3 + 1.5 * iqr

stats = pd.Series({
    "Mean": x.mean(),
    "Median": x.median(),
    "Skewness": x.skew(),
    "Q1": q1,
    "Q3": q3,
    "IQR": iqr,
    "Lower fence": lower,
    "Upper fence": upper,
    "Outliers below lower fence": int((x < lower).sum()),
    "Outliers above upper fence": int((x > upper).sum()),
}).round(2)
print(stats)

corr = done_b[["order_amount", "discount_pct", "quantity", "delivery_minutes"]].corr()
print(corr.round(3))

# largest absolute correlation among distinct pairs
pairs = corr.abs().where(~np.eye(len(corr), dtype=bool)).stack()
a, b = pairs.idxmax()
print("Largest |corr|:", a, "&", b, "=", round(corr.loc[a, b], 3))


Rows: 11110 | Total: 2998668.8
Mean                          269.91
Median                        135.00
Skewness                        4.61
Q1                             64.80
Q3                            297.00
IQR                           232.20
Lower fence                  -283.50
Upper fence                   645.30
Outliers below lower fence      0.00
Outliers above upper fence    897.00
dtype: float64
                  order_amount  discount_pct  quantity  delivery_minutes
order_amount             1.000        -0.095     0.087            -0.003
discount_pct            -0.095         1.000    -0.000            -0.010
quantity                 0.087        -0.000     1.000            -0.012
delivery_minutes        -0.003        -0.010    -0.012             1.000
Largest |corr|: order_amount & discount_pct = -0.095


## Observation:
Mean, Median, Skewness, the quartiles, the 1.5×IQR fences and the number of outliers beyond each fence are observedf from completed order_amount.

The correlation largest in absolute value is order_amount with discount_pct at about −0.095 os confirmed.

## The four questions: 
1. What is this data about? 
    The data explains about the indian customers who have orders in a instamart, which include the products category of Gorcery, Dairy, Elecronoics, Snacks and Personal Care.

2. What do its distributions and statistics look like? 
    The distribution is strongly right-skewed (skewness is ≈4.61), Which tells mean is greated than median(mean = 269.91 and median = 135.00). Most of the order fall between 0-500 range in order_amount, which tell that most of the customers have ordered > 500, and there are only few high value purchase occured.

3. Are there outliers or anomalies? 
    The upper 1.5×IQR fence is 645.30, and 897 orders exceed it. The lower fence is negative, so no orders fall below it. All outliers are unusually large orders a few high value purchase which pull the mean upward.

4. What relationships exist between variables?
    The largest correlation in absolute value is order_amount with discount_pct (≈ −0.095), slightly larger than order_amount with quantity (+0.087). Both are very weak, close to zero. A correlation of this size does not establish causation, so we cannot conclude that discounts change order amounts.